In [2]:
import torch
import h5py

def check_shape_match(pt_file_path, h5_file_path, h5_dataset_name):
    try:
        # 1. Đọc file .pt và lấy shape
        pt_data = torch.load(pt_file_path)
        
        # Nếu file .pt lưu một dictionary (thường là state_dict), bạn cần chỉ định key của tensor cần check.
        # Ở đây giả sử file .pt lưu trực tiếp 1 tensor.
        if isinstance(pt_data, dict):
            print("File .pt là một dictionary. Vui lòng trích xuất đúng Tensor trước khi lấy shape.")
            # Ví dụ: pt_shape = tuple(pt_data['your_tensor_key'].shape)
            return False
            
        pt_shape = tuple(pt_data.shape)
        
        # 2. Đọc file .h5 và lấy shape
        with h5py.File(h5_file_path, 'r') as h5_file:
            # Kiểm tra xem dataset_name có tồn tại trong file .h5 không
            if h5_dataset_name not in h5_file:
                print(f"Không tìm thấy dataset '{h5_dataset_name}' trong file {h5_file_path}")
                print("Các datasets hiện có:", list(h5_file.keys()))
                return False
                
            h5_data = h5_file[h5_dataset_name]
            h5_shape = tuple(h5_data.shape)
        
        # 3. So sánh
        is_match = (pt_shape == h5_shape)
        
        print(f"Shape của file .pt: {pt_shape}")
        print(f"Shape của file .h5 (dataset '{h5_dataset_name}'): {h5_shape}")
        
        if is_match:
            print("=> KẾT QUẢ: Shape trùng khớp!")
        else:
            print("=> KẾT QUẢ: Shape KHÔNG trùng khớp!")
            
        return is_match

    except Exception as e:
        print(f"Đã xảy ra lỗi: {e}")
        return False

# ==========================================
# CÁCH SỬ DỤNG
# ==========================================
pt_path = "/home/uit2023/LuuTru/Thuchd/dataset/NSCLC/feature_256/pt_files/TCGA-18-3416-01Z-00-DX1.EDAD3DEB-8A86-4F34-82EC-6691BFDE900D.pt"
h5_path = "/home/uit2023/LuuTru/Thuchd/dataset/coords_BRCA_CESC_NSCLC/coords_NSCLC/TCGA-18-3416-01Z-00-DX1.EDAD3DEB-8A86-4F34-82EC-6691BFDE900D.h5"
dataset_name = "coords" # Thay đổi tên này cho đúng với dataset của bạn

check_shape_match(pt_path, h5_path, dataset_name)

Shape của file .pt: (4464, 768)
Shape của file .h5 (dataset 'coords'): (5447, 2)
=> KẾT QUẢ: Shape KHÔNG trùng khớp!


False

In [2]:
import torch

/mmlab_students/storageStudents/nguyenvd/Thuchd/clam_latest_bak/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:

a= torch.load("/home/uit2023/LuuTru/Thuchd/CATE_Main_clustering/models/prompt_feats_titan/brca_concepts_titan.pt")
print(type(a))
b= [feat.to("cuda") for feat in a]
print(type(b))

<class 'list'>
<class 'list'>


In [16]:
print(len(a))
print(len(b))
print(a[3].shape)
print(b[0].shape)

6
6
torch.Size([88, 768])
torch.Size([110, 768])


In [6]:
c= torch.load("/mmlab_students/storageStudents/nguyenvd/Thuchd/dataset/BRCA/feature_256/pt_files/TCGA-A2-A0CU-01Z-00-DX1.5B77D21C-A497-478B-9752-3730322AD9ED.pt")
print(c.shape)

torch.Size([4976, 768])


In [ ]:
import tarfile

def gia_nen_targz(file_path, output_path):
    try:
        # Mở file tar.gz với chế độ 'r:gz' (đọc và giải nén gzip)
        with tarfile.open(file_path, "r:gz") as tar:
            
            # Giải nén toàn bộ nội dung vào thư mục output_path
            tar.extractall(path=output_path)
            
            print(f"Đã giải nén thành công file: {file_path}")
            
    except FileNotFoundError:
        print("Lỗi: Không tìm thấy file cần giải nén.")
    except tarfile.ReadError:
        print("Lỗi: File bị hỏng hoặc không phải định dạng .tar.gz")

# --- Ví dụ sử dụng ---
# Thay đổi tên file và thư mục mong muốn của bạn ở đây
file_dau_vao = "/home/uit2023/LuuTru/Thuchd/CATE_Main_clustering/heatmaps/demo/slides/gdc_download_20260131_073015.983140.tar.gz" 
thu_muc_dau_ra = "/home/uit2023/LuuTru/Thuchd/CATE_Main_clustering/heatmaps/demo/slides/"

gia_nen_targz(file_dau_vao, thu_muc_dau_ra)

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score

# =======================
# CONFIG
# =======================
ROOT_DIR = "/home/uit2023/LuuTru/Thuchd/CATE_Main_clustering/results/RCC_final_sim001_alpha05_cross1_sym1_s1"   # chứa split_{i}_results.pkl và splits_{i}.csv
FOLDS = [0, 1, 2, 3, 4]
OUT_CSV = "summary_test_metrics.csv"
# =======================


def load_pkl(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


def read_ids_from_splits(csv_path: str, split_name: str):
    """
    Assumption (TCGA-style):
      splits_i.csv có cột train/val/test, mỗi cột là list id (có NaN)
    """
    df = pd.read_csv(csv_path)
    if split_name not in df.columns:
        raise ValueError(f"'{split_name}' not in splits columns: {list(df.columns)}")
    return [str(x) for x in df[split_name].dropna().tolist()]


def squeeze_prob(p):
    # (1,C)->(C,), (C,)->(C,)
    return np.asarray(p).squeeze().ravel().astype(float)


def stack_probs(probs_list):
    if len(probs_list) == 0:
        return None
    if probs_list[0].size == 1:
        return np.asarray([p.item() for p in probs_list], dtype=float)  # (N,)
    return np.vstack(probs_list)  # (N,C)


def infer_pred(y_prob):
    if y_prob.ndim == 1:
        # scalar prob => binary threshold
        return (y_prob >= 0.5).astype(int)
    return y_prob.argmax(axis=1)


def auc_safe(y_true, y_prob):
    """
    - Binary: roc_auc_score(y_true, prob_class1)
    - Multiclass: roc_auc_score(..., multi_class="ovr")
    Nếu điều kiện không đủ (vd thiếu lớp) -> NaN
    """
    try:
        if y_prob is None:
            return np.nan

        y_true = np.asarray(y_true)
        uniq = np.unique(y_true)

        if y_prob.ndim == 1:
            # scalar prob: chỉ tính nếu đúng 2 lớp
            if len(uniq) == 2:
                return float(roc_auc_score(y_true, y_prob))
            return np.nan

        C = y_prob.shape[1]
        if C == 2:
            if len(uniq) == 2:
                return float(roc_auc_score(y_true, y_prob[:, 1]))
            return np.nan

        # multiclass
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr"))
    except Exception:
        return np.nan


def compute_metrics_from_ids(obj: dict, ids: list):
    """
    obj: dict[id] -> {"label": int, "prob": np.ndarray, ...}
    ids: list id từ splits csv
    """
    y_true, probs = [], []
    missing = 0

    for sid in ids:
        if sid not in obj:
            missing += 1
            continue
        item = obj[sid]
        y_true.append(int(item["label"]))
        probs.append(squeeze_prob(item["prob"]))

    if len(y_true) == 0:
        return {
            "n": 0,
            "missing_in_pkl": missing,
            "labels_true": "",
            "acc": np.nan,
            "auc": np.nan,
            "f1_macro": np.nan,
            "kappa": np.nan,
        }

    y_true = np.asarray(y_true, dtype=int)
    y_prob = stack_probs(probs)
    y_pred = infer_pred(y_prob)

    return {
        "n": int(len(y_true)),
        "missing_in_pkl": int(missing),
        "labels_true": ",".join(map(str, np.unique(y_true).tolist())),
        "acc": float(accuracy_score(y_true, y_pred)),
        "auc": float(auc_safe(y_true, y_prob)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }


def main():
    rows = []

    for fold in FOLDS:
        pkl_path = os.path.join(ROOT_DIR, f"split_{fold}_results.pkl")
        csv_path = os.path.join(ROOT_DIR, f"splits_{fold}.csv")

        obj = load_pkl(pkl_path)

        # --- TCGA-style: tính TEST dựa trên splits csv ---
        test_ids = read_ids_from_splits(csv_path, "test")
        overlap = len(set(test_ids).intersection(set(obj.keys())))

        if overlap == 0:
            # fallback: nhiều experiment lưu pkl chỉ gồm test và không khớp splits
            # -> dùng toàn bộ key trong pkl như test
            print(f"[Fold {fold}] WARNING: overlap(test_ids, pkl_keys)=0 -> dùng toàn bộ pkl keys như TEST")
            test_ids = list(obj.keys())

        met = compute_metrics_from_ids(obj, test_ids)

        rows.append({
            "folds": fold,
            "test_acc": met["acc"],
            "test_auc": met["auc"],
            "test_f1_macro": met["f1_macro"],
            "test_kappa": met["kappa"],
        })

    df = pd.DataFrame(rows).sort_values("folds").reset_index(drop=True)

    # mean/std (chỉ cột số)
    num_cols = ["test_n", "missing_in_pkl", "test_acc", "test_auc", "test_f1_macro", "test_kappa"]
    mean_row = {"folds": "mean", "labels_true": "", **df[num_cols].mean(numeric_only=True).to_dict()}
    std_row  = {"folds": "std",  "labels_true": "", **df[num_cols].std(numeric_only=True, ddof=1).to_dict()}

    df_out = pd.concat([df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

    out_path = os.path.join(ROOT_DIR, OUT_CSV)
    df_out.to_csv(out_path, index=False)
    print("Saved:", out_path)
    print(df_out)


if __name__ == "__main__":
    main()

Saved: /home/uit2023/LuuTru/Thuchd/CATE_Main_clustering/results/RCC_final_sim001_alpha05_cross1_sym1_s1/summary_test_metrics.csv
  folds     test_n  missing_in_pkl labels_true  test_acc  test_auc  \
0     0  88.000000             0.0       0,1,2  0.954545  0.998222   
1     1  90.000000             0.0       0,1,2  0.966667  0.999471   
2     2  98.000000             0.0       0,1,2  0.959184  0.981832   
3     3  94.000000             0.0       0,1,2  0.968085  1.000000   
4     4  93.000000             0.0       0,1,2  0.935484  0.975246   
5  mean  92.600000             0.0              0.956793  0.990954   
6   std   3.847077             0.0              0.013132  0.011588   

   test_f1_macro  test_kappa  
0       0.948270    0.923278  
1       0.972147    0.943408  
2       0.958047    0.931264  
3       0.964002    0.944192  
4       0.930003    0.887545  
5       0.954494    0.925937  
6       0.016226    0.023172  


In [1]:
import os, pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score

# =======================
# CONFIG
# =======================
BASE_DIR = "/mmlab_students/storageStudents/nguyenvd/Thuchd/DISC-MIL/results"   # folder cha chứa nhiều folder experiment
FOLDS = [0, 1, 2, 3, 4]
PER_FOLDER_OUT = "summary_test_metrics.csv"
MASTER_OUT = "ALL_FOLDERS_summary.csv"
RECURSIVE = True
# =======================


def load_pkl(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def read_ids_from_splits(csv_path: str, split_name: str = "test"):
    df = pd.read_csv(csv_path)
    if split_name not in df.columns:
        raise ValueError(f"'{split_name}' not in columns: {list(df.columns)}")
    return [str(x) for x in df[split_name].dropna().tolist()]

def squeeze_prob(p):
    return np.asarray(p).squeeze().ravel().astype(float)

def stack_probs(probs_list):
    if len(probs_list) == 0:
        return None
    if probs_list[0].size == 1:
        return np.asarray([p.item() for p in probs_list], dtype=float)  # (N,)
    return np.vstack(probs_list)  # (N,C)

def infer_pred(y_prob):
    if y_prob.ndim == 1:
        return (y_prob >= 0.5).astype(int)
    return y_prob.argmax(axis=1)

def auc_safe(y_true, y_prob):
    try:
        if y_prob is None:
            return np.nan
        uniq = np.unique(y_true)
        if y_prob.ndim == 1:
            return float(roc_auc_score(y_true, y_prob)) if len(uniq) == 2 else np.nan
        C = y_prob.shape[1]
        if C == 2:
            return float(roc_auc_score(y_true, y_prob[:, 1])) if len(uniq) == 2 else np.nan
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr"))
    except Exception:
        return np.nan

def compute_metrics_from_ids(obj: dict, ids: list):
    y_true, probs = [], []
    missing = 0
    for sid in ids:
        if sid not in obj:
            missing += 1
            continue
        item = obj[sid]
        y_true.append(int(item["label"]))
        probs.append(squeeze_prob(item["prob"]))

    if len(y_true) == 0:
        return {
            "n": 0,
            "missing_in_pkl": missing,
            "labels_true": "",
            "acc": np.nan,
            "auc": np.nan,
            "f1_macro": np.nan,
            "kappa": np.nan,
        }

    y_true = np.asarray(y_true, dtype=int)
    y_prob = stack_probs(probs)
    y_pred = infer_pred(y_prob)

    return {
        "n": int(len(y_true)),
        "missing_in_pkl": int(missing),
        "labels_true": ",".join(map(str, np.unique(y_true).tolist())),
        "acc": float(accuracy_score(y_true, y_pred)),
        "auc": float(auc_safe(y_true, y_prob)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }

def folder_has_required_files(folder: str):
    for i in FOLDS:
        if not os.path.exists(os.path.join(folder, f"split_{i}_results.pkl")):
            return False
        if not os.path.exists(os.path.join(folder, f"splits_{i}.csv")):
            return False
    return True

def evaluate_one_folder(folder: str):
    rows = []
    for fold in FOLDS:
        pkl_path = os.path.join(folder, f"split_{fold}_results.pkl")
        csv_path = os.path.join(folder, f"splits_{fold}.csv")

        obj = load_pkl(pkl_path)
        test_ids = read_ids_from_splits(csv_path, "test")

        # sanity overlap; nếu không overlap thì fallback dùng toàn bộ pkl keys
        overlap = len(set(test_ids).intersection(set(obj.keys())))
        if overlap == 0:
            test_ids = list(obj.keys())

        met = compute_metrics_from_ids(obj, test_ids)
        rows.append({
            "folds": fold,
            "test_n": met["n"],
            "missing_in_pkl": met["missing_in_pkl"],
            "labels_true": met["labels_true"],
            "test_acc": met["acc"],
            "test_auc": met["auc"],
            "test_f1_macro": met["f1_macro"],
            "test_kappa": met["kappa"],
        })

    df = pd.DataFrame(rows).sort_values("folds").reset_index(drop=True)
    num_cols = ["test_n", "missing_in_pkl", "test_acc", "test_auc", "test_f1_macro", "test_kappa"]

    mean_row = {"folds": "mean", "labels_true": "", **df[num_cols].mean(numeric_only=True).to_dict()}
    std_row  = {"folds": "std",  "labels_true": "", **df[num_cols].std(numeric_only=True, ddof=1).to_dict()}
    df_out = pd.concat([df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

    out_path = os.path.join(folder, PER_FOLDER_OUT)
    df_out.to_csv(out_path, index=False)
    return df_out, out_path

def iter_candidate_folders(base_dir: str, recursive: bool = True):
    if recursive:
        for root, dirs, files in os.walk(base_dir):
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            yield root
    else:
        for name in os.listdir(base_dir):
            p = os.path.join(base_dir, name)
            if os.path.isdir(p):
                yield p

def fmt_pm(mean, std, decimals=4):
    if mean is None or (isinstance(mean, float) and np.isnan(mean)):
        return ""
    if std is None or (isinstance(std, float) and np.isnan(std)):
        std = 0.0
    return f"{mean:.{decimals}f} ± {std:.{decimals}f}"

def main():
    master_rows = []
    processed = 0

    for folder in iter_candidate_folders(BASE_DIR, RECURSIVE):
        if not folder_has_required_files(folder):
            continue

        try:
            df_out, out_path = evaluate_one_folder(folder)
            processed += 1

            mean = df_out[df_out["folds"] == "mean"].iloc[0]
            std  = df_out[df_out["folds"] == "std"].iloc[0]

            master_rows.append({
                "folder": os.path.relpath(folder, BASE_DIR),
                "test_acc": fmt_pm(float(mean["test_acc"]), float(std["test_acc"]), 4),
                "test_auc": fmt_pm(float(mean["test_auc"]), float(std["test_auc"]), 4),
                "test_f1_macro": fmt_pm(float(mean["test_f1_macro"]), float(std["test_f1_macro"]), 4),
                "test_kappa": fmt_pm(float(mean["test_kappa"]), float(std["test_kappa"]), 4),
                "test_n_mean": f"{float(mean['test_n']):.1f}",
                "per_folder_csv": out_path,
            })

            print(f"[OK] {folder}")
        except Exception as e:
            print(f"[FAIL] {folder}: {e}")

    master = pd.DataFrame(master_rows)

    # sort theo F1 macro mean (tách mean từ chuỗi "x ± y")
    def parse_mean(pm):
        try:
            return float(str(pm).split("±")[0].strip())
        except Exception:
            return -1e9

    if len(master) > 0:
        master["__sort_f1"] = master["test_f1_macro"].apply(parse_mean)
        master = master.sort_values("__sort_f1", ascending=False).drop(columns="__sort_f1")
        master_out_path = os.path.join(BASE_DIR, MASTER_OUT)
        master.to_csv(master_out_path, index=False)
        print(f"\nSaved MASTER summary: {master_out_path}")
        print(master.head(10))
    else:
        print("\nNo valid folders found.")

    print(f"\nProcessed folders: {processed}")

if __name__ == "__main__":
    main()

[OK] /mmlab_students/storageStudents/nguyenvd/Thuchd/DISC-MIL/results/BRCA_Lsym_Lcross_cgrl_abmil_s1
[OK] /mmlab_students/storageStudents/nguyenvd/Thuchd/DISC-MIL/results/BRCA_Lsym_cgrl_abmil_s1
[OK] /mmlab_students/storageStudents/nguyenvd/Thuchd/DISC-MIL/results/BRCA_Lcross_cgrl_abmil_s1
[OK] /mmlab_students/storageStudents/nguyenvd/Thuchd/DISC-MIL/results/BRCA_Ly_cgrl_abmil_s1
[OK] /mmlab_students/storageStudents/nguyenvd/Thuchd/DISC-MIL/results/BRCA_Ly_Lsym_cgrl_abmil_s1
[OK] /mmlab_students/storageStudents/nguyenvd/Thuchd/DISC-MIL/results/BRCA_Ly_Lcross_cgrl_abmil_s1

Saved MASTER summary: /mmlab_students/storageStudents/nguyenvd/Thuchd/DISC-MIL/results/ALL_FOLDERS_summary.csv
                           folder         test_acc         test_auc  \
5    BRCA_Ly_Lcross_cgrl_abmil_s1  0.9238 ± 0.0121  0.9738 ± 0.0194   
1         BRCA_Lsym_cgrl_abmil_s1  0.9258 ± 0.0293  0.9533 ± 0.0332   
2       BRCA_Lcross_cgrl_abmil_s1  0.9165 ± 0.0319  0.9588 ± 0.0330   
0  BRCA_Lsym_Lcross_cgrl_